In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from email.utils import parsedate_to_datetime
from dateutil import parser as dateutil_parser
from datetime import timezone

PROCESSED_DIR = "drive/MyDrive/phishing project datasets/processed"
ENRON_PATH = f"{PROCESSED_DIR}/enron_emails_cleaned.csv"
NAZARIO_PATH = f"{PROCESSED_DIR}/nazario5_cleaned.csv"

pd.set_option('display.max_colwidth', 200)

In [ ]:
df_enron = pd.read_csv(ENRON_PATH)
df_naz = pd.read_csv(NAZARIO_PATH)

print(f"Enron:   {len(df_enron):,} rows")
print(f"Nazario: {len(df_naz):,} rows")

print("\nEnron sample date strings:")
print(df_enron['date'].head(3).to_list())
print("\nNazario sample date strings:")
print(df_naz['date'].head(3).to_list())

Enron:   250,610 rows
Nazario: 3,012 rows

Enron sample date strings:
['Mon, 14 May 2001 16:39:00 -0700 (PDT)', 'Fri, 4 May 2001 13:51:00 -0700 (PDT)', 'Wed, 18 Oct 2000 03:00:00 -0700 (PDT)']

Nazario sample date strings:
['Fri, 29 Jun 2001 08:36:09 -0500', 'Fri, 29 Jun 2001 09:37:04 -0500', 'Fri, 29 Jun 2001 08:39:30 -0500']


In [ ]:
assert 'date' in df_enron.columns, "Enron file missing 'date' column."
assert 'date' in df_naz.columns, "Nazario file missing 'date' column."
assert df_enron['date'].notna().any() and df_naz['date'].notna().any()
print("PASS: 'date' column present and populated in both files.")

PASS: 'date' column present and populated in both files.


In [ ]:
def parse_temporal_field(date_str):
    """Parse an RFC-2822-style date string into UTC-normalized and local
    components. Handles both:
      Enron:   'Mon, 14 May 2001 16:39:00 -0700 (PDT)'
      Nazario: 'Fri, 29 Jun 2001 08:36:09 -0500'
    """
    result = {
        'utc_datetime': None, 'utc_date': None, 'utc_time': None,
        'local_date': None, 'local_time': None,
        'utc_offset_minutes': None,
        'day_of_week': None,       # 0=Monday .. 6=Sunday (UTC-based)
        'hour_of_day_utc': None,
        'is_weekend': None,
        'date_parse_success': False,
        'date_parse_error': None,
    }

    if pd.isna(date_str) or str(date_str).strip() == '':
        result['date_parse_error'] = 'missing_date'
        return result

    s = str(date_str).strip()
    dt = None

    # Primary parser: stdlib, fast, handles RFC-2822 incl. trailing "(PDT)" comments
    try:
        dt = parsedate_to_datetime(s)
    except Exception:
        dt = None

    # Fallback: dateutil, more forgiving of malformed/non-standard strings
    if dt is None or pd.isna(dt):
        try:
            dt = dateutil_parser.parse(s, fuzzy=True)
        except Exception as e:
            result['date_parse_error'] = f"{type(e).__name__}: {e}"
            return result

    if dt.tzinfo is None:
        # Can't safely convert to UTC without an offset — record what we can
        result['local_date'] = dt.date().isoformat()
        result['local_time'] = dt.time().isoformat()
        result['date_parse_error'] = 'naive_datetime_no_tz'
        return result

    dt_utc = dt.astimezone(timezone.utc)

    result['utc_datetime'] = dt_utc.isoformat()
    result['utc_date'] = dt_utc.date().isoformat()
    result['utc_time'] = dt_utc.time().isoformat()
    result['local_date'] = dt.date().isoformat()
    result['local_time'] = dt.time().isoformat()
    result['utc_offset_minutes'] = dt.utcoffset().total_seconds() / 60
    result['day_of_week'] = dt_utc.weekday()
    result['hour_of_day_utc'] = dt_utc.hour
    result['is_weekend'] = dt_utc.weekday() >= 5
    result['date_parse_success'] = True
    return result


def add_temporal_features(df, date_col='date'):
    parsed = df[date_col].apply(parse_temporal_field)
    parsed_df = pd.DataFrame(list(parsed))
    return pd.concat([df.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)

In [ ]:
df_enron_temporal = add_temporal_features(df_enron, date_col='date')

success_rate = df_enron_temporal['date_parse_success'].mean()
print(f"Enron date parse success rate: {success_rate:.2%}")
df_enron_temporal[['date', 'utc_datetime', 'utc_date', 'utc_time', 'utc_offset_minutes', 'day_of_week']].head(5)

Enron date parse success rate: 100.00%


,date,utc_datetime,utc_date,utc_time,utc_offset_minutes,day_of_week
0,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",2001-05-14T23:39:00+00:00,2001-05-14,23:39:00,-420.0,0
1,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",2001-05-04T20:51:00+00:00,2001-05-04,20:51:00,-420.0,4
2,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",2000-10-18T10:00:00+00:00,2000-10-18,10:00:00,-420.0,2
3,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",2000-10-23T13:13:00+00:00,2000-10-23,13:13:00,-420.0,0
4,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",2000-08-31T12:07:00+00:00,2000-08-31,12:07:00,-420.0,3


In [ ]:
assert df_enron_temporal['date_parse_success'].mean() > 0.95, \
    "Low parse success — inspect df_enron_temporal[~df_enron_temporal.date_parse_success]"

successful = df_enron_temporal[df_enron_temporal['date_parse_success']]
assert successful['utc_datetime'].notna().all()
assert successful['utc_offset_minutes'].between(-720, 720).all(), "Offset out of plausible range."
assert successful['day_of_week'].between(0, 6).all()

print("PASS: Enron temporal fields extracted and within valid ranges.")
print()
print("Failure reasons (if any):")
print(df_enron_temporal.loc[~df_enron_temporal['date_parse_success'], 'date_parse_error'].value_counts())

PASS: Enron temporal fields extracted and within valid ranges.

Failure reasons (if any):
Series([], Name: count, dtype: int64)


In [ ]:
df_naz_temporal = add_temporal_features(df_naz, date_col='date')

success_rate = df_naz_temporal['date_parse_success'].mean()
print(f"Nazario date parse success rate: {success_rate:.2%}")
df_naz_temporal[['date', 'utc_datetime', 'utc_date', 'utc_time', 'utc_offset_minutes', 'day_of_week']].head(5)

Nazario date parse success rate: 98.94%


,date,utc_datetime,utc_date,utc_time,utc_offset_minutes,day_of_week
0,"Fri, 29 Jun 2001 08:36:09 -0500",2001-06-29T13:36:09+00:00,2001-06-29,13:36:09,-300.0,4.0
1,"Fri, 29 Jun 2001 09:37:04 -0500",2001-06-29T14:37:04+00:00,2001-06-29,14:37:04,-300.0,4.0
2,"Fri, 29 Jun 2001 08:39:30 -0500",2001-06-29T13:39:30+00:00,2001-06-29,13:39:30,-300.0,4.0
3,"Fri, 29 Jun 2001 10:35:17 -0500",2001-06-29T15:35:17+00:00,2001-06-29,15:35:17,-300.0,4.0
4,"Fri, 29 Jun 2001 10:40:02 -0500",2001-06-29T15:40:02+00:00,2001-06-29,15:40:02,-300.0,4.0


In [ ]:
assert df_naz_temporal['date_parse_success'].mean() > 0.90, \
    "Low parse success — Nazario dates are more heterogeneous (public phishing corpus), inspect failures."

successful = df_naz_temporal[df_naz_temporal['date_parse_success']]
assert successful['utc_datetime'].notna().all()
assert successful['utc_offset_minutes'].between(-720, 720).all()
assert successful['day_of_week'].between(0, 6).all()

print("PASS: Nazario temporal fields extracted and within valid ranges.")
print()
print("Failure reasons (if any):")
print(df_naz_temporal.loc[~df_naz_temporal['date_parse_success'], 'date_parse_error'].value_counts())

AssertionError: 

In [ ]:
for name, df in [('Enron', df_enron_temporal), ('Nazario', df_naz_temporal)]:
    ok = df[df['date_parse_success']]
    print(f"--- {name} ---")
    print(f"  UTC date range: {ok['utc_date'].min()} to {ok['utc_date'].max()}")
    print(f"  Offset range (min): {ok['utc_offset_minutes'].min()} to {ok['utc_offset_minutes'].max()}")
    print()

--- Enron ---
  UTC date range: 1980-01-01 to 2044-01-04
  Offset range (min): -480.0 to -420.0

--- Nazario ---
  UTC date range: 1992-07-28 to 2022-12-27
  Offset range (min): -720.0 to 780.0



In [ ]:
enron_dates = pd.to_datetime(df_enron_temporal.loc[df_enron_temporal['date_parse_success'], 'utc_date'])
naz_dates = pd.to_datetime(df_naz_temporal.loc[df_naz_temporal['date_parse_success'], 'utc_date'])

overlap = (enron_dates.max() >= naz_dates.min()) and (naz_dates.max() >= enron_dates.min())
print(f"Enron range:   {enron_dates.min().date()} → {enron_dates.max().date()}")
print(f"Nazario range: {naz_dates.min().date()} → {naz_dates.max().date()}")
print(f"Date ranges overlap: {overlap}")

if not overlap:
    print("\nWARNING: Enron and Nazario date ranges do NOT overlap.")
    print("If you use raw calendar date/year as a model feature, the model can trivially")
    print("separate classes by date alone rather than learning genuine phishing signal.")

Enron range:   1980-01-01 → 2044-01-04
Nazario range: 1992-07-28 → 2022-12-27
Date ranges overlap: True


In [ ]:
# True valid range for any real-world UTC offset (all standard timezones on Earth)
VALID_OFFSET_MIN = -720   # UTC-12:00 (Baker/Howland Island)
VALID_OFFSET_MAX = 840    # UTC+14:00 (Kiribati, Line Islands)

# Corpus-specific plausible date windows (generous buffers around known collection periods)
ENRON_DATE_MIN = pd.Timestamp('1997-01-01', tz='UTC')   # a couple years before earliest real Enron mail
ENRON_DATE_MAX = pd.Timestamp('2003-01-01', tz='UTC')   # corpus was collected in 2002; buffer past collapse

NAZARIO_DATE_MIN = pd.Timestamp('1992-01-01', tz='UTC')  # before this, public email/phishing essentially didn't exist
NAZARIO_DATE_MAX = pd.Timestamp('2023-01-01', tz='UTC')  # buffer past this Kaggle dataset's known compilation date

In [ ]:
def flag_temporal_anomalies(df, date_min, date_max,
                             offset_min=VALID_OFFSET_MIN, offset_max=VALID_OFFSET_MAX):
    """Flag rows with implausible parsed dates or impossible UTC offsets.
    Does NOT drop rows — nulls the derived temporal fields and records why,
    so the row (and its text content) remains usable elsewhere in the pipeline.
    """
    df = df.copy()
    df['temporal_anomaly'] = False
    df['temporal_anomaly_reason'] = None

    only_parsed = df['date_parse_success']

    utc_dt = pd.to_datetime(df['utc_datetime'], errors='coerce', utc=True)

    bad_date = only_parsed & ((utc_dt < date_min) | (utc_dt > date_max))
    bad_offset = only_parsed & (
        (df['utc_offset_minutes'] < offset_min) | (df['utc_offset_minutes'] > offset_max)
    )

    df.loc[bad_date, 'temporal_anomaly'] = True
    df.loc[bad_date, 'temporal_anomaly_reason'] = 'implausible_date'

    # if both bad, keep a combined reason rather than overwriting
    both = bad_date & bad_offset
    df.loc[both, 'temporal_anomaly_reason'] = 'implausible_date+offset'
    df.loc[bad_offset & ~bad_date, 'temporal_anomaly'] = True
    df.loc[bad_offset & ~bad_date, 'temporal_anomaly_reason'] = 'implausible_offset'

    # Null out derived temporal fields for anomalous rows so they can't
    # silently pollute model input — raw 'date' string is preserved untouched.
    temporal_cols = ['utc_datetime', 'utc_date', 'utc_time', 'local_date', 'local_time',
                      'utc_offset_minutes', 'day_of_week', 'hour_of_day_utc', 'is_weekend']
    df.loc[df['temporal_anomaly'], temporal_cols] = np.nan

    return df

In [ ]:
df_enron_temporal = flag_temporal_anomalies(df_enron_temporal, ENRON_DATE_MIN, ENRON_DATE_MAX)
df_naz_temporal = flag_temporal_anomalies(df_naz_temporal, NAZARIO_DATE_MIN, NAZARIO_DATE_MAX)

for name, df in [('Enron', df_enron_temporal), ('Nazario', df_naz_temporal)]:
    n_anom = df['temporal_anomaly'].sum()
    print(f"--- {name} ---")
    print(f"  Anomalous rows: {n_anom:,} ({n_anom/len(df):.2%})")
    if n_anom > 0:
        print(df.loc[df['temporal_anomaly'], 'temporal_anomaly_reason'].value_counts())
    print()

--- Enron ---
  Anomalous rows: 361 (0.14%)
temporal_anomaly_reason
implausible_date    361
Name: count, dtype: int64

--- Nazario ---
  Anomalous rows: 0 (0.00%)



/tmp/ipykernel_842/2869152133.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[df['temporal_anomaly'], temporal_cols] = np.nan


In [ ]:
temporal_cols = ['utc_datetime', 'utc_date', 'utc_time', 'local_date', 'local_time',
                  'utc_offset_minutes', 'day_of_week', 'hour_of_day_utc', 'is_weekend']

for name, df in [('Enron', df_enron_temporal), ('Nazario', df_naz_temporal)]:
    anom = df[df['temporal_anomaly']]
    clean = df[~df['temporal_anomaly'] & df['date_parse_success']]

    # anomalous rows should have nulled temporal fields
    assert anom[temporal_cols].isna().all().all(), f"{name}: anomalous rows still have temporal values."

    # clean rows should be fully within bounds
    if len(clean) > 0:
        assert clean['utc_offset_minutes'].between(VALID_OFFSET_MIN, VALID_OFFSET_MAX).all(), \
            f"{name}: a 'clean' row has an out-of-bounds offset."

    # original raw date string should be untouched even for anomalies
    assert df.loc[df['temporal_anomaly'], 'date'].notna().all(), \
        f"{name}: raw 'date' string was lost during anomaly handling."

    print(f"PASS ({name}): anomalies flagged, temporal fields nulled, raw date preserved, clean rows in-bounds.")

PASS (Enron): anomalies flagged, temporal fields nulled, raw date preserved, clean rows in-bounds.
PASS (Nazario): anomalies flagged, temporal fields nulled, raw date preserved, clean rows in-bounds.


In [ ]:
for name, df in [('Enron', df_enron_temporal), ('Nazario', df_naz_temporal)]:
    ok = df[df['date_parse_success'] & ~df['temporal_anomaly']]
    print(f"--- {name} (clean rows only) ---")
    print(f"  UTC date range: {ok['utc_date'].min()} to {ok['utc_date'].max()}")
    print(f"  Offset range (min): {ok['utc_offset_minutes'].min()} to {ok['utc_offset_minutes'].max()}")
    print()

--- Enron (clean rows only) ---
  UTC date range: 1997-01-01 to 2002-12-21
  Offset range (min): -480.0 to -420.0

--- Nazario (clean rows only) ---
  UTC date range: 1992-07-28 to 2022-12-27
  Offset range (min): -720.0 to 780.0



In [ ]:
# Base folder path in your Google Drive
folder_path = '/content/drive/My Drive/phishing project datasets/processed/'

# Output filepaths with the '_temporal' suffix
ENRON_OUT = folder_path + 'enron_emails_cleaned_temporal.csv'
NAZ_OUT = folder_path + 'nazario5_cleaned_temporal.csv'

# Save the DataFrames to CSV
df_enron_temporal.to_csv(ENRON_OUT, index=False)
df_naz_temporal.to_csv(NAZ_OUT, index=False)

# Display saving confirmation and anomaly stats
print(f"Saved: {ENRON_OUT}  ({len(df_enron_temporal):,} rows, {df_enron_temporal['temporal_anomaly'].sum():,} flagged anomalous)")
print(f"Saved: {NAZ_OUT}  ({len(df_naz_temporal):,} rows, {df_naz_temporal['temporal_anomaly'].sum():,} flagged anomalous)")

Saved: /content/drive/My Drive/phishing project datasets/processed/enron_emails_cleaned_temporal.csv  (250,610 rows, 361 flagged anomalous)
Saved: /content/drive/My Drive/phishing project datasets/processed/nazario5_cleaned_temporal.csv  (3,012 rows, 0 flagged anomalous)


In [ ]:
df_enron_temporal.head()

,file,message,message_id,date,from,to,subject,cc,bcc,x_from,...,local_date,local_time,utc_offset_minutes,day_of_week,hour_of_day_utc,is_weekend,date_parse_success,date_parse_error,temporal_anomaly,temporal_anomaly_reason
0,allen-p/_sent_mail/1.,"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nConte...",<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,NaN,NaN,NaN,Phillip K Allen,...,2001-05-14,16:39:00,-420.0,0.0,23.0,False,True,None,False,None
1,allen-p/_sent_mail/10.,"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\n...",<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,NaN,NaN,Phillip K Allen,...,2001-05-04,13:51:00,-420.0,4.0,20.0,False,True,None,False,None
2,allen-p/_sent_mail/100.,"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct 2000 03:00:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: leah.arsdall@enron.com\nSubject: Re: test\nMime-Version: ...",<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,NaN,NaN,Phillip K Allen,...,2000-10-18,03:00:00,-420.0,2.0,10.0,False,True,None,False,None
3,allen-p/_sent_mail/1000.,"Message-ID: <13505866.1075863688222.JavaMail.evans@thyme>\nDate: Mon, 23 Oct 2000 06:13:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: randall.gay@enron.com\nSubject: \nMime-Version: 1.0\nCont...",<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,NaN,NaN,NaN,Phillip K Allen,...,2000-10-23,06:13:00,-420.0,0.0,13.0,False,True,None,False,None
4,allen-p/_sent_mail/1001.,"Message-ID: <30922949.1075863688243.JavaMail.evans@thyme>\nDate: Thu, 31 Aug 2000 05:07:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: greg.piper@enron.com\nSubject: Re: Hello\nMime-Version: 1...",<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,NaN,NaN,Phillip K Allen,...,2000-08-31,05:07:00,-420.0,3.0,12.0,False,True,None,False,None
